# Vanilla 3DGS — instrumented experiment (event + metric logging)

Runs **upstream graphdeco 3DGS** (stock rasterizer, `--eval`) through the
`experiments/` harness, instrumented by `scripts/vanilla_densify_log.py` so it
writes `densify_events.pt` — the same artifact SGDense runs produce — then scores
it on the unified metric path.

This mirrors `SGDense_3DGS_Experiment.ipynb`; the difference is that it targets
the *stock* 3DGS codebase rather than the seg-aware fork, and adds no
segmentation terms, so densification is pure photometric ADC.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get this repo (harness + `sgdense` + scripts) and dependencies

Upstream 3DGS is cloned separately in cell 5 by the harness; here we only need the
SGDense-3DGS checkout for `experiments/`, `sgdense/` and `scripts/`.

In [ ]:
%cd /content
!rm -rf gaussian-splatting
!git clone -q https://github.com/satyaba/gaussian-splatting.git
%cd gaussian-splatting
!git submodule update --init --recursive
!pip install -q plyfile opencv-python tqdm joblib lpips torchmetrics tensorboard
print("repo ready")

## 3. GPU / CUDA check

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# The rasterizer needs nvcc's CUDA to match torch.version.cuda; fail fast otherwise.
import re, shutil, subprocess, warnings
def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    try:
        out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    except Exception:
        return None
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None

torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print("\nCUDA toolchain check:")
print(f"  torch : {torch.__version__} (built for CUDA {torch_cuda})")
print(f"  nvcc  : {nvcc_cuda or 'NOT FOUND on PATH'}")
if nvcc_cuda is None:
    warnings.warn("nvcc not found on PATH -- CUDA extensions cannot be built.")
elif torch_cuda is None:
    warnings.warn("This PyTorch build has no CUDA.")
elif torch_cuda.split('.')[0] != nvcc_cuda.split('.')[0]:
    raise RuntimeError(f"CUDA major mismatch: torch {torch_cuda} vs nvcc {nvcc_cuda}.")
else:
    print("  OK: toolchain consistent.")

## 4. Configure paths + run id

In [ ]:
import os, sys

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"   # adjust if needed
REPO_ROOT = "/content/gaussian-splatting"
DATASET = "office_0/Sequence_1"

os.environ["HOME_DIR"] = HOME_DIR
os.environ["DATASET_ROOT"] = os.path.join(HOME_DIR, "Dataset")
os.environ["SGDENSE_REPO"] = REPO_ROOT
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

SOURCE = os.path.join(os.environ["DATASET_ROOT"], DATASET)
RUN_ID = "vanilla_30k_logged"
ITERS = 30000
MODEL_DIR = os.path.join(HOME_DIR, "output", RUN_ID)
os.makedirs(MODEL_DIR, exist_ok=True)
print("source :", SOURCE)
print("model  :", MODEL_DIR)

## 5. Clone + build upstream graphdeco (stock rasterizer)

`prepare()` clones `graphdeco-inria/gaussian-splatting` into
`/content/gaussian-splatting-vanilla`, initializes only the training submodules,
and swaps the installed `diff_gaussian_rasterization` to the **stock** one.

In [ ]:
from experiments.methods import get_method
from experiments.runner import prepare

method = get_method("vanilla_3dgs")
wd = prepare(method, workdir_root="/content")
print("upstream workdir:", wd)

## 6. Instrument + train

We copy `scripts/vanilla_densify_log.py` into the upstream clone and run its
`train.py` under that instrumentation. It monkeypatches `GaussianModel` so
clone / split / prune / opacity-reset fire a `DensifyTracker`, then writes
`<MODEL_DIR>/densify_events.pt` at interpreter exit.

The train args deliberately match the harness `vanilla_3dgs` invocation
(`--eval --disable_viewer --quiet`) so this run is comparable to `vanilla_30k`.

In [ ]:
import shutil, subprocess, sys, os, time

src = os.path.join(REPO_ROOT, "scripts", "vanilla_densify_log.py")
dst = os.path.join(str(wd), "vanilla_densify_log.py")
shutil.copy(src, dst)

events_path = os.path.join(MODEL_DIR, "densify_events.pt")

code = ("import vanilla_densify_log, runpy, sys; "
        "sys.argv = ['train.py'] + sys.argv[1:]; "
        "runpy.run_path('train.py', run_name='__main__')")
train_args = ["-s", SOURCE, "-m", MODEL_DIR,
              "--eval", "--disable_viewer", "--quiet", "--iterations", str(ITERS)]
cmd = [sys.executable, "-c", code] + train_args

env = dict(os.environ,
           SGDENSE_REPO=REPO_ROOT,
           DENSIFY_EVENTS_OUT=events_path,
           PYTHONPATH=os.pathsep.join(
               [str(wd), REPO_ROOT] +
               [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]))

t0 = time.time()
print("$", " ".join(cmd))
subprocess.run(cmd, cwd=str(wd), env=env, check=True)
train_seconds = time.time() - t0
print("train time: %.2f min" % (train_seconds / 60))

# record wall-clock so the metrics row's "Train time (min)" column is filled
from experiments.runner import _write_train_time
_write_train_time(MODEL_DIR, train_seconds, cmd)

## 7. Extract metrics (same code path as the harness)

`extract_one` evaluates the model with this repo's seg-aware rasterizer (its RGB
path is stock 3DGS), so it renders the vanilla PLY correctly and appends a row to
`$HOME_DIR/output/metrics.csv`. Note this swaps the rasterizer back to the
seg-aware one first.

In [ ]:
from experiments.metrics import extract_one

csv_path = os.path.join(HOME_DIR, "output", "metrics.csv")
row = extract_one(MODEL_DIR, iteration=-1, run_id=RUN_ID,
                  name="Upstream graphdeco 3DGS + event log, --eval split.",
                  csv_path=csv_path)
row

## 8. Load + visualize the densification events

In [ ]:
from sgdense import load_events
import pandas as pd

log = load_events(events_path)
print("events:", len(log.events), "| summary rows:", len(log.summary))
pd.DataFrame(log.summary).tail(10)

In [ ]:
from sgdense.viz import plot_timeline, plot_prune_breakdown, save_report

plot_timeline(log)
plot_prune_breakdown(log)

report = os.path.join(MODEL_DIR, "densify_report.html")
save_report(log, report, config={"run_id": RUN_ID, "method": "vanilla_3dgs"})
print("wrote", report)

## 9. Compare with SGDense

Load any SGDense run's log (it has per-Gaussian class info; vanilla's classes are
all `-1`) and overlay the timelines.

In [ ]:
# sg = load_events(os.path.join(HOME_DIR, "output", "run38_30k", "densify_events.pt"))
# plot_timeline(sg)
# print("SGDense events:", len(sg.events))